# E1 only: conditional hidden-state comparison

This notebook runs only the nested E1 analysis: N, N+H, O, O+H, and H. It reuses frozen Experiment 1 activation shards and the selected-layer artifact already stored in Google Drive. It does not load the language model, extract activations, refit the original layer sweep, run interventions, or build the paper.

A GPU is not required for this analysis. A Colab high-RAM runtime is recommended because the frozen activation shards occupy roughly 2 GiB before analysis workspaces are allocated.

## 1. Load the repository

Set `REPOSITORY_REF` to the branch or tag containing the E1 implementation before running this cell. If the notebook is already running from a repository checkout, that checkout is used directly.

In [2]:
import os
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path


def load_secret(name: str, prompt: str) -> str:
    token = os.environ.get(name, "").strip()
    if not token:
        try:
            from google.colab import userdata

            token = (userdata.get(name) or "").strip()
        except Exception:
            token = ""
    if not token:
        token = getpass(prompt).strip()
    if not token:
        raise RuntimeError(f"{name} is required.")
    return token


GITHUB_TOKEN = load_secret("GITHUB_TOKEN", "GitHub token (hidden): ")
HF_TOKEN = load_secret("HF_TOKEN", "Hugging Face token (hidden): ")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ.setdefault("HF_HOME", "/content/huggingface")

askpass_path = Path(tempfile.gettempdir()) / "math_error_github_askpass.sh"
askpass_path.write_text(
    (
        "#!/bin/sh\n"
        'case "$1" in\n'
        "*Username*) echo x-access-token ;;\n"
        '*Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        "esac\n"
    ),
    encoding="utf-8",
)
askpass_path.chmod(0o700)


def github_git_env() -> dict[str, str]:
    environment = os.environ.copy()
    environment.update(
        {
            "GITHUB_TOKEN": GITHUB_TOKEN,
            "GIT_ASKPASS": str(askpass_path),
            "GIT_ASKPASS_REQUIRE": "force",
            "GIT_TERMINAL_PROMPT": "0",
        }
    )
    return environment


REPOSITORY = "https://github.com/sagnikc395/tracing-mathematical-error-detection-in-language-models.git"
AUTHENTICATED_REPOSITORY = REPOSITORY.replace("https://", "https://x-access-token@", 1)
REPOSITORY_REF = "main"  # Change if E1 is on another branch.
REPOSITORY_NAME = "tracing-mathematical-error-detection-in-language-models"

working_directory = Path.cwd()
if (working_directory / "pyproject.toml").exists():
    repository_directory = working_directory
elif (working_directory.parent / "pyproject.toml").exists():
    repository_directory = working_directory.parent
else:
    repository_directory = Path("/content") / REPOSITORY_NAME
    if not (repository_directory / "pyproject.toml").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPOSITORY_REF,
                AUTHENTICATED_REPOSITORY,
                str(repository_directory),
            ],
            check=True,
            env=github_git_env(),
        )

os.chdir(repository_directory)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    check=True,
)
from huggingface_hub import HfApi

hf_identity = HfApi().whoami(token=HF_TOKEN)
print(f"Repository: {repository_directory}")
print(f"Hugging Face: authenticated as {hf_identity['name']}")
print(f"Python: {sys.version.split()[0]}")

Repository: /content/tracing-mathematical-error-detection-in-language-models
Hugging Face: authenticated as sagnikcw2
Python: 3.13.15


## 2. Mount Drive and identify the frozen inputs

Edit `EXPERIMENT1_DIR` if your previous notebook used a different Drive folder. The directory must directly contain both `activation_shards/` and `probes/directions.npz`.

In [3]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/math-error-tracing")
EXPERIMENT1_DIR = (
    DRIVE_ROOT / "artifacts/qwen2.5-math-1.5b-a100-bf16"
)
DATA_PATH = DRIVE_ROOT / "data/processbench.jsonl"
OUTPUT_DIR = DRIVE_ROOT / "artifacts/experiment3-extended"

print(f"Frozen Experiment 1 directory: {EXPERIMENT1_DIR}")
print(f"ProcessBench data: {DATA_PATH}")
print(f"E1 output directory: {OUTPUT_DIR / 'conditional_hidden_state'}")

Mounted at /content/drive
Frozen Experiment 1 directory: /content/drive/MyDrive/math-error-tracing/artifacts/qwen2.5-math-1.5b-a100-bf16
ProcessBench data: /content/drive/MyDrive/math-error-tracing/data/processbench.jsonl
E1 output directory: /content/drive/MyDrive/math-error-tracing/artifacts/experiment3-extended/conditional_hidden_state


## 3. Validate the shard set

Every shard must have an activation array (`.npy`), aligned metadata (`.csv`), and manifest (`.json`). The selected layer is read from the frozen probe artifact. The dataset hash is checked against the extraction identity before any model is fit.

In [4]:
import hashlib
import json

import numpy as np

SHARD_DIR = EXPERIMENT1_DIR / "activation_shards"
DIRECTION_PATH = EXPERIMENT1_DIR / "probes/directions.npz"
IDENTITY_PATH = EXPERIMENT1_DIR / "extraction_identity.json"

for required_path in (SHARD_DIR, DIRECTION_PATH, IDENTITY_PATH, DATA_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Required frozen input is missing: {required_path}")

array_paths = sorted(SHARD_DIR.glob("shard_*.npy"))
if not array_paths:
    raise FileNotFoundError(f"No activation shards found under {SHARD_DIR}")
incomplete = [
    path.stem
    for path in array_paths
    if not path.with_suffix(".csv").exists()
    or not path.with_suffix(".json").exists()
]
if incomplete:
    raise RuntimeError(f"Incomplete shard triplets: {incomplete[:5]}")

identity = json.loads(IDENTITY_PATH.read_text())
dataset_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if identity.get("dataset_sha256") != dataset_sha256:
    raise RuntimeError(
        "The ProcessBench file does not match the dataset used for shard extraction."
    )
if identity.get("dtype") != "bfloat16":
    raise RuntimeError(f"Expected the frozen BF16 run, found {identity.get('dtype')!r}")

directions = np.load(DIRECTION_PATH)
selected_layer = int(directions["selected_layer"])
shard_bytes = sum(path.stat().st_size for path in array_paths)
print(f"Complete shard triplets: {len(array_paths)}")
print(f"Activation-array size: {shard_bytes / 2**30:.2f} GiB")
print(f"Frozen selected layer: {selected_layer}")
print(json.dumps(identity, indent=2))

Complete shard triplets: 34
Activation-array size: 2.07 GiB
Frozen selected layer: 23
{
  "dataset_sha256": "447f0a4b35c5747a9f9a3dab1e70d43f71efd501497b8cec668b71337099784a",
  "model": "Qwen/Qwen2.5-Math-1.5B-Instruct",
  "dtype": "bfloat16",
  "max_length": 2048
}


## 4. Resolve the E1 configuration

Only filesystem paths are changed from `configs/experiment3.yaml`. The C grid, bootstrap count, seed, confidence level, feature definitions, and practical AUROC margin remain frozen in the repository configuration.

In [5]:
import yaml

with Path("configs/experiment3.yaml").open(encoding="utf-8") as handle:
    e1_config = yaml.safe_load(handle)
e1_config["experiment1_dir"] = str(EXPERIMENT1_DIR)
e1_config["data_path"] = str(DATA_PATH)
e1_config["output_dir"] = str(OUTPUT_DIR)
e1_config["counterfactual_patching"]["pairs_path"] = str(
    DRIVE_ROOT / "data/counterfactual_pairs.jsonl"
)

CONFIG_PATH = Path("/content/conditional_hidden_state.yaml")
CONFIG_PATH.write_text(
    yaml.safe_dump(e1_config, sort_keys=False), encoding="utf-8"
)
print(CONFIG_PATH.read_text())

seed: 42
experiment1_dir: /content/drive/MyDrive/math-error-tracing/artifacts/qwen2.5-math-1.5b-a100-bf16
data_path: /content/drive/MyDrive/math-error-tracing/data/processbench.jsonl
output_dir: /content/drive/MyDrive/math-error-tracing/artifacts/experiment3-extended
bootstrap_samples: 1000
confidence_level: 0.95
model:
  name: Qwen/Qwen2.5-Math-1.5B-Instruct
  device: auto
  dtype: float16
  max_length: 2048
transition_probe:
  c_values:
  - 0.01
  - 0.1
  - 1.0
  - 10.0
  max_iter: 2000
conditional_hidden_state:
  practical_auroc_margin: 0.02
  tfidf_min_df: 2
  tfidf_max_features: 20000
boundary_control:
  save_every: 100
  batch_size: 1
counterfactual_patching:
  pairs_path: /content/drive/MyDrive/math-error-tracing/data/counterfactual_pairs.jsonl
  template_size: 160
  batch_size: 1



## 5. Run E1

This is the only experiment command in the notebook. It loads the frozen shards, fits all five conditions, evaluates the untouched test partition, and writes its outputs directly to Drive. No existing Experiment 1 artifact is modified.

In [6]:
from datetime import datetime, timezone
from time import monotonic

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / "conditional_hidden_state.log"
command = [
    sys.executable,
    "-m",
    "tracing_math.experiment3.cli",
    "--config",
    str(CONFIG_PATH),
    "fit-conditional-hidden-state",
]
print("$", " ".join(command), flush=True)
started = monotonic()
started_at = datetime.now(timezone.utc).isoformat()
with LOG_PATH.open("a", encoding="utf-8", buffering=1) as log_file:
    log_file.write(f"\n[{started_at}] START {' '.join(command)}\n")
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    return_code = process.wait()
    log_file.write(f"[{datetime.now(timezone.utc).isoformat()}] EXIT {return_code}\n")
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
print(f"E1 complete in {(monotonic() - started) / 60:.1f} minutes")
print(f"Log: {LOG_PATH}")

$ /usr/bin/python3 -m tracing_math.experiment3.cli --config /content/conditional_hidden_state.yaml fit-conditional-hidden-state
{
  "status": "complete",
  "analysis_scope": "post_hoc",
  "selected_layer": 23,
  "conditions": [
    "N",
    "N+H",
    "O",
    "O+H",
    "H"
  ],
  "test_traces": 669,
  "practical_auroc_margin": 0.02,
  "config_sha256": "c4b87dfc613b82c1b6b1ea12a1512777beee77670712d24168a468b7424b8356",
  "artifacts": {
    "metrics": "metrics.csv",
    "selection": "validation_selection.csv",
    "predictions": "test_predictions.csv",
    "paired_intervals": "paired_differences.csv",
    "feature_blocks": "feature_blocks.json",
    "result_record": "result_record.md"
  },
  "updated_at": "2026-09-02T17:51:59.412493+00:00"
}
E1 complete in 14.9 minutes
Log: /content/drive/MyDrive/math-error-tracing/artifacts/experiment3-extended/conditional_hidden_state.log


## 6. Inspect and verify the outputs

The metric table contains absolute held-out results. The paired table contains literal left-minus-right differences; negative log-loss and Brier-score differences favor the model with hidden features.

In [7]:
import pandas as pd
from IPython.display import Markdown, display

RESULT_DIR = OUTPUT_DIR / "conditional_hidden_state"
required_outputs = (
    "metrics.csv",
    "validation_selection.csv",
    "test_predictions.csv",
    "paired_differences.csv",
    "feature_blocks.json",
    "resolved_config.json",
    "result_record.md",
    "summary.json",
)
missing_outputs = [name for name in required_outputs if not (RESULT_DIR / name).exists()]
if missing_outputs:
    raise RuntimeError(f"E1 completed without required outputs: {missing_outputs}")

metrics = pd.read_csv(RESULT_DIR / "metrics.csv")
paired = pd.read_csv(RESULT_DIR / "paired_differences.csv")
summary = json.loads((RESULT_DIR / "summary.json").read_text())
if summary.get("status") != "complete":
    raise RuntimeError(f"Unexpected E1 status: {summary.get('status')!r}")
if set(metrics["condition"]) != {"N", "N+H", "O", "O+H", "H"}:
    raise RuntimeError("The metric artifact does not contain all five conditions.")

metric_columns = [
    "condition",
    "auroc",
    "average_precision",
    "log_loss",
    "brier_score",
    "error_exact",
    "correct_rejection",
    "process_f1",
    "error_within_1_accuracy",
    "error_within_2_accuracy",
    "complete_accuracy",
]
display(metrics[metric_columns].round(4))
display(
    paired[
        paired["metric"].isin(
            ["auroc", "average_precision", "log_loss", "brier_score", "process_f1"]
        )
    ][
        [
            "comparison",
            "metric",
            "estimate",
            "ci_low",
            "ci_high",
            "favorable_direction",
        ]
    ].round(4)
)
display(Markdown((RESULT_DIR / "result_record.md").read_text()))
print(f"Resolved config hash: {summary['config_sha256']}")
print(f"Saved outputs: {RESULT_DIR}")

,condition,auroc,average_precision,log_loss,brier_score,error_exact,correct_rejection,process_f1,error_within_1_accuracy,error_within_2_accuracy,complete_accuracy
0,N,0.8105,0.7635,0.5333,0.1784,0.1667,0.2869,0.2109,0.4907,0.7014,0.2093
1,N+H,0.8692,0.8478,0.4536,0.1484,0.2824,0.6709,0.3975,0.5810,0.7130,0.4200
2,O,0.8740,0.8645,0.4403,0.1428,0.1690,0.8945,0.2843,0.4537,0.6227,0.4260
3,O+H,0.9088,0.8933,0.3848,0.1197,0.2940,0.9072,0.4441,0.5833,0.7454,0.5112
4,H,0.8677,0.8455,0.4574,0.1499,0.3194,0.5232,0.3967,0.6273,0.7361,0.3916


,comparison,metric,estimate,ci_low,ci_high,favorable_direction
0,N+H - N,auroc,0.0587,0.0414,0.0745,higher
1,N+H - N,average_precision,0.0843,0.0556,0.1134,higher
2,N+H - N,log_loss,-0.0798,-0.1038,-0.0548,lower
3,N+H - N,brier_score,-0.0300,-0.0391,-0.0204,lower
6,N+H - N,process_f1,0.1866,0.1397,0.2340,higher
10,O+H - O,auroc,0.0348,0.0242,0.0451,higher
11,O+H - O,average_precision,0.0288,0.0168,0.0424,higher
12,O+H - O,log_loss,-0.0555,-0.0773,-0.0337,lower
13,O+H - O,brier_score,-0.0231,-0.0308,-0.0156,lower
16,O+H - O,process_f1,0.1598,0.0972,0.2179,higher


# E1 conditional hidden-state result record

This record was generated from held-out, trace-paired predictions under resolved configuration
`c4b87dfc613b82c1b6b1ea12a1512777beee77670712d24168a468b7424b8356`.

- N+H minus N AUROC: +0.0587 (95% interval
  [+0.0414, +0.0745]).
- N+H minus N log loss: -0.0798 (95% interval
  [-0.1038, -0.0548]); negative favors N+H.
- Frozen practical AUROC margin: 0.0200; the interval does not rule out an increment at the frozen practical margin.

Interpret the ranking, proper-score, and localization rows separately. This post-hoc result does not
by itself provide an untouched confirmatory replication.


Resolved config hash: c4b87dfc613b82c1b6b1ea12a1512777beee77670712d24168a468b7424b8356
Saved outputs: /content/drive/MyDrive/math-error-tracing/artifacts/experiment3-extended/conditional_hidden_state


In [8]:
import shutil

GITHUB_BRANCH = "main"
GIT_AUTHOR_NAME = "sagnikc395"
GIT_AUTHOR_EMAIL = "sagnikchatterjee607@gmail.com"
MAX_GITHUB_FILE_BYTES = 95 * 1024 * 1024

staged_before = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if staged_before:
    raise RuntimeError(
        "The repository already has staged changes; clear them before publishing E1."
    )

publish_directory = (
    repository_directory
    / "artifacts/experiment3_extended/conditional_hidden_state"
)
published_files = []
for filename in required_outputs:
    source = RESULT_DIR / filename
    destination = publish_directory / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    published_files.append(destination)

oversized = [
    path for path in published_files if path.stat().st_size > MAX_GITHUB_FILE_BYTES
]
if oversized:
    names = ", ".join(
        str(path.relative_to(repository_directory)) for path in oversized
    )
    raise RuntimeError(f"Files exceed the safe GitHub limit: {names}")

published_paths = [
    str(path.relative_to(repository_directory)) for path in published_files
]
subprocess.run(["git", "add", "-f", "--", *published_paths], check=True)
staged = subprocess.run(
    ["git", "diff", "--cached", "--name-only", "--", *published_paths],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

if not staged:
    print("E1 results are already up to date on this checkout.")
else:
    subprocess.run(["git", "config", "user.name", GIT_AUTHOR_NAME], check=True)
    subprocess.run(["git", "config", "user.email", GIT_AUTHOR_EMAIL], check=True)
    subprocess.run(
        [
            "git",
            "commit",
            "-m",
            "Add conditional hidden-state results",
            "--",
            *published_paths,
        ],
        check=True,
    )
    subprocess.run(
        ["git", "pull", "--rebase", AUTHENTICATED_REPOSITORY, GITHUB_BRANCH],
        check=True,
        env=github_git_env(),
    )
    subprocess.run(
        ["git", "push", AUTHENTICATED_REPOSITORY, f"HEAD:{GITHUB_BRANCH}"],
        check=True,
        env=github_git_env(),
    )
    print(f"Published {len(published_paths)} E1 files to {GITHUB_BRANCH}.")

Published 8 E1 files to main.
